In [11]:
import pandas as pd
import numpy as np
from difflib import get_close_matches

# -------- STEP 1: LOAD FILES -------- #
wdi_df = pd.read_csv("WDICSV.csv", low_memory=False)
hdi_df = pd.read_excel("HDI.xlsx", sheet_name="Table 1. HDI")
happiness_df = pd.read_excel("World Hapiness.xlsx")

# -------- STEP 2: FILTER & RESHAPE WDI -------- #
indicator_codes = {
    "NY.GDP.PCAP.CD": "GDP per capita",
    "SE.XPD.TOTL.GD.ZS": "Education spending (% of GDP)",
    "SH.XPD.CHEX.GD.ZS": "Health spending (% of GDP)",
    "GB.XPD.RSDV.GD.ZS": "R&D spending (% of GDP)",
    "EG.ELC.ACCS.ZS": "Access to electricity (% of population)",
    "SL.UEM.TOTL.ZS": "Unemployment rate (% of labor force)",
    "SP.DYN.LE00.IN": "Life expectancy at birth",
    "SP.POP.GROW": "Population growth (annual %)"
}
wdi_df = wdi_df[wdi_df["Indicator Code"].isin(indicator_codes)]
wdi_long = wdi_df.melt(
    id_vars=["Country Name", "Indicator Code"],
    value_vars=[str(y) for y in range(2000, 2024)],
    var_name="Year",
    value_name="Value"
)
wdi_pivot = wdi_long.pivot_table(
    index=["Country Name", "Year"],
    columns="Indicator Code",
    values="Value"
).reset_index()
wdi_pivot.rename(columns=indicator_codes, inplace=True)
wdi_pivot.rename(columns={"Country Name": "Country"}, inplace=True)
wdi_pivot["Year"] = pd.to_numeric(wdi_pivot["Year"], errors="coerce")

# -------- STEP 3: CLEAN HDI -------- #
hdi_data = hdi_df.iloc[7:].copy()
hdi_data.columns = [
    "HDI rank", "Country", "HDI", "ignore1", "Life expectancy",
    "ignore2", "Expected years of schooling", "ignore3", "Mean years of schooling",
    "ignore4", "GNI per capita", "ignore5", "GNI rank minus HDI rank", "ignore6", "HDI rank (2022)"
]
hdi_clean = hdi_data[[
    "Country", "HDI", "Life expectancy", "Expected years of schooling",
    "Mean years of schooling", "GNI per capita"
]].dropna(subset=["Country"])
for col in hdi_clean.columns[1:]:
    hdi_clean[col] = pd.to_numeric(hdi_clean[col], errors='coerce')
hdi_clean["Year"] = 2023  # Static year for HDI

# -------- STEP 4: CLEAN HAPPINESS DATA -------- #
happiness_df.rename(columns=lambda x: x.strip().lower().replace(" ", "_"), inplace=True)
happiness_df.rename(columns={"country": "Country"}, inplace=True)
happiness_df["Year"] = pd.to_numeric(happiness_df["year"], errors="coerce")

# -------- STEP 5: FUZZY MERGE SUPPORT -------- #
def fuzzy_merge(left, right, left_on, right_on, threshold=0.9):
    left = left.copy()
    mapping = {}
    for country in left[left_on].unique():
        match = get_close_matches(country, right[right_on].unique(), n=1, cutoff=threshold)
        if match:
            mapping[country] = match[0]
    left[left_on] = left[left_on].replace(mapping)
    return left

wdi_pivot = fuzzy_merge(wdi_pivot, happiness_df, "Country", "Country")
hdi_clean = fuzzy_merge(hdi_clean, wdi_pivot, "Country", "Country")
happiness_df = fuzzy_merge(happiness_df, wdi_pivot, "Country", "Country")

# -------- STEP 6: MERGE -------- #
merged_df = pd.merge(wdi_pivot, hdi_clean, on=["Country", "Year"], how="outer")
merged_df = pd.merge(merged_df, happiness_df, on=["Country", "Year"], how="outer")

# -------- STEP 7: FILTER & CLEAN -------- #
merged_df = merged_df.dropna(subset=["GDP per capita", "HDI", "happiness_score"], how="all")
merged_df.replace(['..', '—', '-', ''], np.nan, inplace=True)

# -------- STEP 8: SAVE -------- #
merged_df.to_csv("merged_development_data_full.csv", index=False)
print("✅ Saved merged dataset with shape:", merged_df.shape)


✅ Saved merged dataset with shape: (6400, 24)


In [12]:
import pandas as pd
dfHDI=pd.read_csv("merged_development_data_full.csv")
print(dfHDI.head())

       Country  Year  Access to electricity (% of population)  \
0  Afghanistan  2000                                      4.4   
1  Afghanistan  2001                                      9.3   
2  Afghanistan  2002                                     14.1   
3  Afghanistan  2003                                     19.0   
4  Afghanistan  2004                                     23.8   

   R&D spending (% of GDP)  GDP per capita  Education spending (% of GDP)  \
0                      NaN      174.930991                            NaN   
1                      NaN      138.706822                            NaN   
2                      NaN      178.954088                            NaN   
3                      NaN      198.871116                            NaN   
4                      NaN      221.763654                            NaN   

   Health spending (% of GDP)  Unemployment rate (% of labor force)  \
0                         NaN                                 7.935   
1   

In [16]:
import pandas as pd
import numpy as np

# -------------------------
# STEP 1: Load the Data
# -------------------------
df = pd.read_csv("merged_development_data_full.csv")

# -------------------------
# STEP 2: Normalize Column Names
# -------------------------
df.columns = df.columns.astype(str).str.strip().str.lower().str.replace('\n', ' ').str.replace(' ', '_')

# -------------------------
# STEP 3: Clean Invalid Strings
# -------------------------
# Replace common non-numeric placeholders with NaN
df.replace(to_replace=['..', '—', '-', '', 'N/A', 'na', 'NaN'], value=np.nan, inplace=True)

# -------------------------
# STEP 4: Convert Columns to Numeric (Safe)
# -------------------------
# Skip 'country' and other known string columns
non_numeric_cols = ['country', 'region', 'subregion', 'indicator_name']  # update as needed
for col in df.columns:
    if col not in non_numeric_cols and df[col].ndim == 1:
        try:
            df[col] = pd.to_numeric(df[col], errors='coerce')
        except Exception as e:
            print(f"⚠️ Skipping {col} due to conversion error: {e}")

# -------------------------
# STEP 5: Drop Duplicate Rows
# -------------------------
df = df.drop_duplicates()

# -------------------------
# STEP 6: Optional — Summary Stats and NA Report
# -------------------------
print("\n✅ Cleaned Data Shape:", df.shape)
print("\n📊 Null Count (Top 20 Columns):")
print(df.isnull().sum().sort_values(ascending=False).head(20))

# Optional: Save cleaned version for modeling
df.to_csv("preprocessed_development_data.csv", index=False)
print("\n💾 Saved as: preprocessed_development_data.csv")



✅ Cleaned Data Shape: (6400, 24)

📊 Null Count (Top 20 Columns):
mean_years_of_schooling                    6192
gni_per_capita                             6192
hdi                                        6192
life_expectancy                            6191
expected_years_of_schooling                6191
healthy_life_expectancy                    5034
perceptions_of_corruption                  5034
region                                     5033
social_support                             5033
freedom_to_make_life_choices               5033
gdp_per_capita                             5033
happiness_score                            5033
generosity                                 5033
year                                       5033
r&d_spending_(%_of_gdp)                    3675
education_spending_(%_of_gdp)              2071
health_spending_(%_of_gdp)                  955
unemployment_rate_(%_of_labor_force)        827
access_to_electricity_(%_of_population)     251
gdp_per_capita        

In [19]:
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.inspection import permutation_importance
import warnings
warnings.filterwarnings("ignore")

# -------- LOAD DATA -------- #
df = pd.read_csv("merged_development_data_full.csv")
df.columns = df.columns.str.strip().str.replace('\n', ' ')

# -------- CLEANING -------- #
df.replace(['..', '—', '-', '', np.nan], np.nan, inplace=True)

# Convert all but 'country' to numeric
for col in df.columns:
    if col.lower() != 'country':
        df[col] = pd.to_numeric(df[col], errors='coerce')

# -------- FEATURE GROUPS -------- #
common_features = [
    "Access to electricity (% of population)",
    "Education spending (% of GDP)",
    "Health spending (% of GDP)",
    "R&D spending (% of GDP)",
    "social_support",
    "freedom_to_make_life_choices",
    "generosity",
    "perceptions_of_corruption"
]

targets = {
    "HDI": common_features,
    "GDP per capita": common_features,
    "happiness_score": common_features
}

# -------- PREPROCESS FUNCTION -------- #
def preprocess(df, features, target, log_transform=False):
    df_sub = df[features + [target]].copy()
    df_sub = df_sub.dropna(subset=[target])
    df_sub[features] = KNNImputer(n_neighbors=3).fit_transform(df_sub[features])

    if log_transform:
        df_sub[target] = np.log1p(df_sub[target])

    X = StandardScaler().fit_transform(df_sub[features])
    y = df_sub[target].values

    return train_test_split(X, y, test_size=0.2, random_state=42), df_sub[features], df_sub[target]

# -------- MODEL TRAINING FUNCTION -------- #
def train_model(X_train, X_test, y_train, y_test, param_grid=None):
    model = MLPRegressor(
        activation='relu',
        solver='adam',
        max_iter=2000,
        early_stopping=True,
        validation_fraction=0.15,
        random_state=42
    )

    if param_grid:
        model = GridSearchCV(model, param_grid, cv=5, scoring='r2', verbose=0)

    model.fit(X_train, y_train)

    best_model = model.best_estimator_ if param_grid else model

    train_preds = best_model.predict(X_train)
    test_preds = best_model.predict(X_test)

    train_r2 = r2_score(y_train, train_preds)
    test_r2 = r2_score(y_test, test_preds)
    test_mse = mean_squared_error(y_test, test_preds)

    return best_model, test_mse, test_r2, train_r2

# -------- FEATURE IMPORTANCE -------- #
def show_feature_importance(model, X, y, feature_names):
    results = permutation_importance(model, X, y, n_repeats=10, random_state=42)
    importance = pd.Series(results.importances_mean, index=feature_names).sort_values(ascending=False)
    print("\n🔍 Feature Importance:")
    print(importance)

# -------- EXECUTION -------- #
param_grid = {
    'hidden_layer_sizes': [(64, 64), (128,), (32, 16)],
    'alpha': [0.0001, 0.001, 0.01]
}

for target, features in targets.items():
    print(f"\n========= {target.upper()} =========")

    log_gdp = (target == "GDP per capita")
    (X_train, X_test, y_train, y_test), X_full, y_full = preprocess(df, features, target, log_transform=log_gdp)

    model, test_mse, test_r2, train_r2 = train_model(X_train, X_test, y_train, y_test, param_grid=param_grid)
    
    print(f"✅ Train R² = {train_r2:.4f} | Test R² = {test_r2:.4f} | Test MSE = {test_mse:.4f}")
    show_feature_importance(model, X_test, y_test, features)



========= HDI =========
✅ Train R² = 0.6778 | Test R² = 0.3437 | Test MSE = 0.0131

🔍 Feature Importance:
Access to electricity (% of population)    2.081470
social_support                             1.852936
Health spending (% of GDP)                 0.494355
R&D spending (% of GDP)                    0.449746
generosity                                 0.291634
perceptions_of_corruption                  0.265243
Education spending (% of GDP)              0.215068
freedom_to_make_life_choices               0.015704
dtype: float64

========= GDP PER CAPITA =========
✅ Train R² = 0.7701 | Test R² = 0.7621 | Test MSE = 0.5688

🔍 Feature Importance:
Access to electricity (% of population)    0.729854
social_support                             0.133659
R&D spending (% of GDP)                    0.075014
perceptions_of_corruption                  0.064622
freedom_to_make_life_choices               0.032493
Health spending (% of GDP)                 0.024211
Education spending (% of GDP)   